In [32]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- 1. SETTINGS ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "../Models/cnn_bilstm_attention.pth"
VIDEO_PATH = "../Datasets/Processed_Data/Front/W001/W001S03F_01.mp4" 
LANDMARK_PATH = "../Datasets/normalized_landmarks/Front/W001/W001S03F_01.npy"
OUTPUT_VIDEO = "gradcam_aligned_final.mp4"

# --- 2. ARCHITECTURE (Restored from 4.1 Notebook) ---
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)
    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return (weights * x).sum(dim=1)

class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, input_dim=387, num_classes=401):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(input_dim, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        self.lstm = nn.LSTM(input_size=256, hidden_size=256, batch_first=True, bidirectional=True)
        self.attention = Attention(256)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        # Input x: (Batch, Time, Features)
        x = x.permute(0, 2, 1)      # To (Batch, Features, Time)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)      # To (Batch, Time, Features)
        x, _ = self.lstm(x)
        x = self.attention(x)
        return self.fc(x)

In [33]:
# --- 3. GRAD-CAM ENGINE ---
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hooks = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]
        self.hooks.append(self.target_layer.register_forward_hook(forward_hook))
        self.hooks.append(self.target_layer.register_full_backward_hook(backward_hook))

    def generate_heatmap(self, input_tensor):
        self.model.eval()
        output = self.model(input_tensor)
        idx = output.argmax(dim=1).item()
        self.model.zero_grad()
        output[0, idx].backward()
        weights = torch.mean(self.gradients, dim=2, keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1).squeeze()
        cam = torch.relu(cam)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy()

In [34]:
# --- 4. EXECUTION & VISUALIZATION ---
# ... (model loading code as before) ...

# 1. Generate the raw heatmap from the CNN
data = np.load(LANDMARK_PATH) # Shape (60, 387)
input_tensor = torch.tensor(data).unsqueeze(0).float().to(DEVICE)
raw_heatmap = cam_engine.generate_heatmap(input_tensor) # This returns size 30 or 15

# 2. RESIZE HEATMAP TO MATCH VIDEO LENGTH (60 FRAMES)
# This maps the 30 attention points across all 60 frames smoothly
x_original = np.linspace(0, 1, len(raw_heatmap))
x_target = np.linspace(0, 1, 60)
heatmap = np.interp(x_target, x_original, raw_heatmap)

# 3. Open Video and Writer
cap = cv2.VideoCapture(VIDEO_PATH)
w_px = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h_px = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w_px, h_px))

frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame_idx >= 60: 
        break

    # Now heatmap[frame_idx] will always work because we interpolated to 60
    attn = heatmap[frame_idx]
    
    # Color mapping: Blue (Low) -> Red (High)
    color = (int(255 * (1 - attn)), int(255 * attn * 0.5), int(255 * attn))

    # DRAW LANDMARKS
    current_lms = data[frame_idx].reshape(-1, 3)
    for lm in current_lms:
        if lm[0] != 0 or lm[1] != 0:
            x_pos = int(lm[0] * w_px)
            y_pos = int(lm[1] * h_px)
            cv2.circle(frame, (x_pos, y_pos), 3, color, -1)

    # UI OVERLAYS
    cv2.putText(frame, f"Model Attention: {attn:.2f}", (30, 60), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    cv2.rectangle(frame, (0, h_px-20), (int(w_px * attn), h_px), color, -1)

    out.write(frame)
    frame_idx += 1

cap.release()
out.release()
print(f"File Saved: {OUTPUT_VIDEO}")

File Saved: gradcam_aligned_final.mp4
